# Datathon 2026 — Phần 1: Câu hỏi Trắc nghiệm
10 câu × 2 điểm = **20 điểm**

In [1]:
import pandas as pd

BASE = '/kaggle/input/datasets/oncngpho/dataset/'

orders      = pd.read_csv(BASE + 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(BASE + 'order_items.csv', low_memory=False)
products    = pd.read_csv(BASE + 'products.csv')
customers   = pd.read_csv(BASE + 'customers.csv')
returns     = pd.read_csv(BASE + 'returns.csv')
web_traffic = pd.read_csv(BASE + 'web_traffic.csv')
payments    = pd.read_csv(BASE + 'payments.csv')
geography   = pd.read_csv(BASE + 'geography.csv')
sales       = pd.read_csv(BASE + 'sales.csv', parse_dates=['Date'])

print('All files loaded.')

All files loaded.


## Q1 — Trung vị số ngày giữa hai lần mua liên tiếp
Tính từ `orders.csv`, chỉ lấy khách hàng có **nhiều hơn 1 đơn hàng**.

In [2]:
gaps = (orders.sort_values('order_date')
              .groupby('customer_id')['order_date']
              .apply(lambda x: x.diff().dt.days.dropna().tolist()))
multi = gaps[gaps.apply(len) > 0]
all_gaps = [g for lst in multi for g in lst]
median_gap = pd.Series(all_gaps).median()
print(f'Median inter-order gap: {median_gap:.1f} days')
print('=> ANS: C) 144 ngày')

Median inter-order gap: 144.0 days
=> ANS: C) 144 ngày


## Q2 — Phân khúc sản phẩm có tỷ suất lợi nhuận gộp trung bình cao nhất
Công thức: `(price - cogs) / price`

In [12]:
# Validate: Check for null/zero prices
print(f'Products with null price: {products["price"].isna().sum()}')
print(f'Products with price = 0: {(products["price"] == 0).sum()}')
print(f'Products with null cogs: {products["cogs"].isna().sum()}')

# Clean data
products_clean = products.dropna(subset=['price', 'cogs'])
products_clean = products_clean[products_clean['price'] > 0]
print(f'\nProducts after cleaning: {len(products_clean)} (removed {len(products) - len(products_clean)})')

products_clean['margin'] = (products_clean['price'] - products_clean['cogs']) / products_clean['price']
seg_margin = products_clean.groupby('segment')['margin'].mean().sort_values(ascending=False)
print(f'\nMargin by segment:')
print(seg_margin)
print(f'\n=> ANS: D) {seg_margin.idxmax()} ({seg_margin.max():.2%})')

Products with null price: 0
Products with price = 0: 0
Products with null cogs: 0

Products after cleaning: 2412 (removed 0)

Margin by segment:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: margin, dtype: float64

=> ANS: D) Standard (31.34%)


## Q3 — Lý do trả hàng phổ biến nhất trong danh mục Streetwear
Join `returns` với `products` theo `product_id`, lọc `category == 'Streetwear'`.

In [4]:
sw = returns.merge(products[['product_id', 'category']], on='product_id')
sw_counts = sw[sw['category'] == 'Streetwear']['return_reason'].value_counts()
print(sw_counts)
print(f'\n=> ANS: B) {sw_counts.idxmax()} ({sw_counts.max():,} lần)')

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

=> ANS: B) wrong_size (7,626 lần)


## Q4 — Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất
Từ `web_traffic.csv`, group by `traffic_source`, tính `mean(bounce_rate)`.

In [5]:
br = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()
print(br)
print(f'\n=> ANS: C) {br.idxmin()} ({br.min():.6f})')

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

=> ANS: C) email_campaign (0.004458)


## Q5 — Tỷ lệ dòng trong `order_items.csv` có áp dụng khuyến mãi
`promo_id` không null.

In [6]:
pct = order_items['promo_id'].notna().mean() * 100
print(f'% rows with promo_id: {pct:.1f}%')
print('=> ANS: C) 39%')

% rows with promo_id: 38.7%
=> ANS: C) 39%


## Q6 — Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất
Join `customers` với `orders`, lọc `age_group` không null.

In [7]:
cust_orders = orders.groupby('customer_id').size().reset_index(name='n_orders')
merged = customers[customers['age_group'].notna()].merge(cust_orders, on='customer_id', how='left')
merged['n_orders'] = merged['n_orders'].fillna(0)
ag = merged.groupby('age_group')['n_orders'].mean().sort_values(ascending=False)
print(ag)
print(f'\n=> ANS: A) {ag.idxmax()} ({ag.max():.2f} đơn/khách)')

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: n_orders, dtype: float64

=> ANS: A) 55+ (5.41 đơn/khách)


## Q7 — Vùng tạo ra tổng doanh thu cao nhất
Join `order_items` → `orders` → `geography`, tính revenue = `unit_price × quantity - discount_amount`.

In [8]:
# Data quality check
print('=== Q7 DATA QUALITY CHECK ===')
print(f'Order status distribution:')
print(orders['order_status'].value_counts())
print(f'\nCancelled orders: {(orders["order_status"] == "cancelled").sum():,}')
print(f'Returned orders: {(orders["order_status"] == "returned").sum():,}')
print(f'Delivered orders: {(orders["order_status"] == "delivered").sum():,}')

# Filter for delivered orders only
oi = order_items.copy()
oi['rev'] = oi['unit_price'] * oi['quantity'] - oi['discount_amount']

# Only include delivered orders
ord_filtered = orders[orders['order_status'] == 'delivered'][['order_id', 'zip']]
print(f'Order items before filter: {len(oi):,}')
print(f'Orders (delivered only): {len(ord_filtered):,}')

ord_geo = ord_filtered.merge(geography[['zip', 'region']], on='zip')
region_rev = oi.merge(ord_geo, on='order_id').groupby('region')['rev'].sum().sort_values(ascending=False)
print(f'\nRevenue by region (DELIVERED ORDERS ONLY):')
print(region_rev)
print(f'\n=> ANS: C) {region_rev.idxmax()} ({region_rev.max():,.0f} VND)')

=== Q7 DATA QUALITY CHECK ===
Order status distribution:
order_status
delivered    516716
cancelled     59462
returned      36142
shipped       13773
paid          13577
created        7275
Name: count, dtype: int64

Cancelled orders: 59,462
Returned orders: 36,142
Delivered orders: 516,716

=== CALCULATING REVENUE (FIXED VERSION) ===
Order items before filter: 714,669
Orders (delivered only): 516,716

Revenue by region (DELIVERED ORDERS ONLY):
region
East       5.826256e+09
Central    3.762670e+09
West       2.929250e+09
Name: rev, dtype: float64

=> ANS: C) East (5,826,256,457 VND)


## Q8 — Phương thức thanh toán phổ biến nhất trong đơn hàng bị huỷ
Lọc `order_status == 'cancelled'`, đếm `payment_method`.

In [14]:
cancelled = orders[orders['order_status'] == 'cancelled']
null_payment = cancelled['payment_method'].isna().sum()
print(f'Total cancelled orders: {len(cancelled):,}')
print(f'Cancelled orders with NULL payment_method: {null_payment:,}')
if null_payment > 0:
    print(f'⚠️ WARNING: {null_payment:,} cancelled orders ({null_payment/len(cancelled)*100:.1f}%) have no payment_method')
else:
    print('✅ All cancelled orders have payment_method data')

cancelled_pm = cancelled['payment_method'].value_counts()
print(f'\nPayment method distribution in cancelled orders:')
print(cancelled_pm)
print(f'\n=> ANS: A) {cancelled_pm.idxmax()} ({cancelled_pm.max():,} lần)')

Total cancelled orders: 59,462
Cancelled orders with NULL payment_method: 0
✅ All cancelled orders have payment_method data

Payment method distribution in cancelled orders:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

=> ANS: A) credit_card (28,452 lần)


## Q9 — Kích thước có tỷ lệ trả hàng cao nhất
Tỷ lệ = số bản ghi trong `returns` / số dòng trong `order_items`, theo từng size (S, M, L, XL).

In [10]:
sizes = ['S', 'M', 'L', 'XL']
ret_size  = returns.merge(products[['product_id', 'size']], on='product_id')
ret_counts = ret_size[ret_size['size'].isin(sizes)]['size'].value_counts()
oi_size   = order_items.merge(products[['product_id', 'size']], on='product_id')
oi_counts  = oi_size[oi_size['size'].isin(sizes)]['size'].value_counts()
rate = (ret_counts / oi_counts).sort_values(ascending=False)
print(rate)
print(f'\n=> ANS: A) {rate.idxmax()} ({rate.max():.4f})')

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64

=> ANS: A) S (0.0565)


## Q10 — Kế hoạch trả góp có giá trị thanh toán trung bình cao nhất
Từ `payments.csv`, group by `installments`, tính `mean(payment_value)`.

In [11]:
inst = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)
print(inst)
print(f'\n=> ANS: C) {inst.idxmax()} kỳ ({inst.max():,.2f})')

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64

=> ANS: C) 6 kỳ (24,446.65)


---
## Tổng kết đáp án

| Câu | Đáp án | Lựa chọn |
|-----|--------|----------|
| Q1  | 144 ngày | **C** |
| Q2  | Standard | **D** |
| Q3  | wrong_size | **B** |
| Q4  | email_campaign | **C** |
| Q5  | ~39% | **C** |
| Q6  | 55+ | **A** |
| Q7  | East | **C** |
| Q8  | credit_card | **A** |
| Q9  | S | **A** |
| Q10 | 6 kỳ | **C** |